# 01 From Scratch — Perceptron

**Status:** Complete

**Learning outcome:** Build Neuron, Layer, and MLP classes on top of the Value autograd engine, then train a tiny network on a toy classification problem with decision boundary visualisation.

## Why This Matters

A single neuron computes `tanh(w·x + b)` — the atomic unit of neural networks. By composing neurons into layers and layers into an MLP, we build the simplest trainable classifier. Training it with gradient descent from our Value engine makes the entire chain from scalar calculus to network training concrete and debuggable.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import math
import random

random.seed(42)
np.random.seed(42)
print("Libraries loaded.")

Libraries loaded.


## Value Class (from previous notebook)

In [2]:
class Value:
    """Scalar value with autograd support (copied from 01_scalar_autograd)."""
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __truediv__(self, other):
        return self * (other ** -1) if isinstance(other, Value) else self * (Value(other) ** -1)

print("Value class loaded.")

Value class loaded.


## Neuron, Layer, MLP

In [3]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


# Test: MLP with 2 inputs, hidden layer of 4, output of 1
model = MLP(2, [4, 1])
print(f"MLP(2, [4, 1]) — {len(model.parameters())} parameters")
out = model([Value(1.0), Value(2.0)])
print(f"Forward pass output: {out.data:.4f}")

MLP(2, [4, 1]) — 17 parameters
Forward pass output: -0.3284


---
**Understanding What a Neuron Computes**

**Plain language:** A neuron is like a panel of judges scoring a contestant. Each judge (weight) watches a different aspect of the performance (input), gives it a score, and those scores are added together with a baseline mood (bias). The total is then squeezed through a "clap-o-meter" that caps how enthusiastic or negative the reaction can be — that's the activation function.

**Building intuition:** Each input $x_i$ is multiplied by a learned weight $w_i$, producing a weighted vote for how important that input is. The bias $b$ shifts the overall tendency before any input is seen. The raw sum $w \cdot x + b$ is a linear combination — it can only carve out flat decision boundaries. The activation function (here `tanh`) bends this linear output into the range $[-1, 1]$, introducing the curvature that lets networks model complex patterns.

**Formal statement:** A neuron computes $y = \sigma\!\bigl(\sum_{i=1}^{n} w_i x_i + b\bigr)$ where $\sigma$ is a nonlinear activation function. With $\sigma = \tanh$, the output is $y = \tanh(w^\top x + b) \in (-1, 1)$. The parameters $\{w_i, b\}$ are adjusted during training to minimise a loss function over the dataset.

---

## Training on a Toy Dataset

We create a simple 2D classification problem (two half-moons) and train our MLP using MSE loss and manual gradient descent.

In [4]:
# Generate toy data: simple 2-class problem
from sklearn.datasets import make_moons
X_np, y_np = make_moons(n_samples=80, noise=0.15, random_state=42)
y_np = y_np * 2 - 1  # Convert to {-1, +1} for tanh output

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X_np[y_np == 1, 0], X_np[y_np == 1, 1], c='blue', label='+1', s=20)
ax.scatter(X_np[y_np == -1, 0], X_np[y_np == -1, 1], c='red', label='-1', s=20)
ax.set_title('Training Data (make_moons)')
ax.legend()
plt.tight_layout(); plt.show()
print(f"Dataset: {len(X_np)} samples, 2 classes")

Dataset: 80 samples, 2 classes


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_49045/2640439775.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [5]:
# Training loop
random.seed(42)
model = MLP(2, [8, 8, 1])  # 2-input, two hidden layers of 8, 1 output
lr = 0.05
losses = []

for epoch in range(50):
    # Forward: compute predictions and MSE loss
    preds = [model(x.tolist()) for x in X_np]
    loss = sum((p - y) ** 2 for p, y in zip(preds, y_np)) * (1.0 / len(preds))

    # Zero grads
    for p in model.parameters():
        p.grad = 0.0

    # Backward
    loss.backward()

    # Update
    for p in model.parameters():
        p.data -= lr * p.grad

    losses.append(loss.data)
    if epoch % 10 == 0:
        acc = sum(1 for p, y in zip(preds, y_np) if (p.data > 0) == (y > 0)) / len(y_np)
        print(f"Epoch {epoch:3d} | Loss: {loss.data:.4f} | Accuracy: {acc:.2%}")

# Final accuracy
preds = [model(x.tolist()) for x in X_np]
acc = sum(1 for p, y in zip(preds, y_np) if (p.data > 0) == (y > 0)) / len(y_np)
print(f"\nFinal accuracy: {acc:.2%}")
assert acc > 0.85, f"Accuracy {acc:.2%} too low"
assert losses[-1] < losses[0], "Loss did not decrease"

Epoch   0 | Loss: 0.8776 | Accuracy: 67.50%


Epoch  10 | Loss: 0.4456 | Accuracy: 83.75%


Epoch  20 | Loss: 0.3494 | Accuracy: 86.25%


Epoch  30 | Loss: 0.3155 | Accuracy: 87.50%


Epoch  40 | Loss: 0.3023 | Accuracy: 88.75%



Final accuracy: 91.25%


---
**Understanding The Training Loop (Forward-Backward-Update)**

**Plain language:** Training a neural network is like a student taking a practice exam, checking the answer key, figuring out which study habits led to mistakes, and then adjusting those habits before the next attempt. Each round through this cycle makes the student a little better. The model "studies" by repeating predict-check-adjust over and over.

**Building intuition:** In the forward pass, every input flows through the network to produce a prediction. The loss function (here, mean squared error) measures how far off those predictions are from the true labels. The backward pass uses the chain rule to assign blame — each parameter gets a gradient telling it how much it contributed to the error and in which direction. The update step nudges every parameter by a small amount (the learning rate) opposite to its gradient, so the loss shrinks. Zeroing gradients before each backward pass prevents stale blame from prior epochs from accumulating.

**Formal statement:** At each epoch the loop executes: (1) forward $\hat{y}_i = f_\theta(x_i)$; (2) loss $\mathcal{L} = \frac{1}{N}\sum_i (\hat{y}_i - y_i)^2$; (3) backward $\frac{\partial \mathcal{L}}{\partial \theta_j}$ for all parameters $\theta_j$ via reverse-mode autodiff; (4) update $\theta_j \leftarrow \theta_j - \eta \frac{\partial \mathcal{L}}{\partial \theta_j}$. This is vanilla gradient descent with learning rate $\eta$.

---

## Decision Boundary Visualisation

In [ ]:
# Decision Boundary Evolution — retrain with snapshots at epochs 0, 25, 49
import copy

random.seed(42)
evo_model = MLP(2, [8, 8, 1])
evo_lr = 0.05
evo_snapshots = {}  # epoch -> list of (w_data, grad) tuples
evo_snapshot_epochs = [0, 25, 49]

# Save a snapshot of all parameter values
def evo_save_snapshot(m):
    return [p.data for p in m.parameters()]

def evo_load_snapshot(m, snap):
    for p, val in zip(m.parameters(), snap):
        p.data = val

# Snapshot at epoch 0 (before any training)
evo_snapshots[0] = evo_save_snapshot(evo_model)

for evo_epoch in range(50):
    evo_preds = [evo_model(x.tolist()) for x in X_np]
    evo_loss = sum((p - y) ** 2 for p, y in zip(evo_preds, y_np)) * (1.0 / len(evo_preds))
    for p in evo_model.parameters():
        p.grad = 0.0
    evo_loss.backward()
    for p in evo_model.parameters():
        p.data -= evo_lr * p.grad
    if evo_epoch + 1 in [25, 50]:  # after update at epoch 24 -> snapshot "25", epoch 49 -> snapshot "50"
        evo_snapshots[evo_epoch + 1] = evo_save_snapshot(evo_model)

# Build decision boundary grid (reuse bounds from earlier)
evo_h = 0.05
evo_x_min, evo_x_max = X_np[:, 0].min() - 0.5, X_np[:, 0].max() + 0.5
evo_y_min, evo_y_max = X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5
evo_xx, evo_yy = np.meshgrid(np.arange(evo_x_min, evo_x_max, evo_h),
                              np.arange(evo_y_min, evo_y_max, evo_h))
evo_grid = np.c_[evo_xx.ravel(), evo_yy.ravel()]

# Plot 1x3 panel
evo_fig, evo_axes = plt.subplots(1, 3, figsize=(15, 4))
evo_titles = ['Epoch 0 (random init)', 'Epoch 25 (mid-training)', 'Epoch 50 (final)']

for evo_ax, (evo_ep, evo_snap), evo_title in zip(evo_axes, sorted(evo_snapshots.items()), evo_titles):
    evo_load_snapshot(evo_model, evo_snap)
    evo_Z = np.array([evo_model(pt.tolist()).data for pt in evo_grid])
    evo_Z = evo_Z.reshape(evo_xx.shape)

    evo_ax.contourf(evo_xx, evo_yy, evo_Z, levels=20, cmap='RdBu', alpha=0.6)
    evo_ax.contour(evo_xx, evo_yy, evo_Z, levels=[0], colors='black', linewidths=2)
    evo_ax.scatter(X_np[y_np == 1, 0], X_np[y_np == 1, 1], c='blue', s=20, edgecolors='k', linewidths=0.5)
    evo_ax.scatter(X_np[y_np == -1, 0], X_np[y_np == -1, 1], c='red', s=20, edgecolors='k', linewidths=0.5)
    evo_ax.set_title(evo_title)
    evo_ax.set_xlim(evo_x_min, evo_x_max)
    evo_ax.set_ylim(evo_y_min, evo_y_max)

evo_fig.suptitle('Decision Boundary Evolution During Training', fontsize=14, y=1.02)
plt.tight_layout()
evo_fig.savefig('../assets/decision_boundary_evolution.png', dpi=100, bbox_inches='tight')
plt.show()
print("Decision boundary evolution saved to ../assets/decision_boundary_evolution.png")

In [6]:
# Decision boundary
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(losses)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss')

# Decision boundary
h = 0.05
x_min, x_max = X_np[:, 0].min() - 0.5, X_np[:, 0].max() + 0.5
y_min, y_max = X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid_points = np.c_[xx.ravel(), yy.ravel()]
Z = np.array([model(p.tolist()).data for p in grid_points])
Z = Z.reshape(xx.shape)

axes[1].contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.6)
axes[1].contour(xx, yy, Z, levels=[0], colors='black', linewidths=2)
axes[1].scatter(X_np[y_np == 1, 0], X_np[y_np == 1, 1], c='blue', s=20, edgecolors='k', linewidths=0.5)
axes[1].scatter(X_np[y_np == -1, 0], X_np[y_np == -1, 1], c='red', s=20, edgecolors='k', linewidths=0.5)
axes[1].set_title(f'Decision Boundary (acc={acc:.0%})')

plt.tight_layout()
plt.savefig('../assets/perceptron_decision_boundary.png', dpi=100)
plt.show()
print("Decision boundary saved.")

Decision boundary saved.


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_49045/1535279281.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
**Understanding Why Nonlinearity is Necessary**

**Plain language:** Imagine stacking several sheets of tracing paper, each with a single straight line drawn on it. No matter how many sheets you stack or rotate, you can only ever see straight-line boundaries when you look through the pile. The activation function is like allowing each sheet to bend and warp — suddenly you can outline curves, loops, and complex shapes. Without it, more layers add nothing new.

**Building intuition:** A linear function composed with another linear function is still linear: $W_2(W_1 x + b_1) + b_2 = W_2 W_1 x + (W_2 b_1 + b_2)$, which collapses to a single matrix multiply plus bias. This means a 100-layer linear network is no more powerful than a 1-layer one. The classic XOR problem — where points at (0,0) and (1,1) belong to one class and (0,1) and (1,0) to another — cannot be separated by any single line. The nonlinear activation between layers lets the network fold and twist the input space so that previously inseparable classes end up on different sides of a boundary.

**Formal statement:** For linear layers $f_i(x) = W_i x + b_i$, the composition $f_k \circ \cdots \circ f_1$ is itself an affine map $x \mapsto Ax + c$ where $A = \prod W_i$. The function class does not grow with depth. Introducing a nonlinear activation $\sigma$ yields $f_k \circ \sigma \circ \cdots \circ \sigma \circ f_1$, which by the universal approximation theorem can approximate any continuous function on a compact set given sufficient width.

---

## Structured Interpretation

1. **Neuron = linear combination + nonlinearity**: Each neuron computes `tanh(w·x + b)`, squashing its output to [-1, 1]. Without tanh, stacking layers would just give another linear function.

2. **Training loop anatomy**: Forward → loss → zero_grad → backward → update. This is the exact same pattern used in PyTorch, just on scalars instead of tensors.

3. **Decision boundary is nonlinear**: With two hidden layers of 8 neurons, the network can learn curved boundaries to separate the half-moon classes. A single perceptron (no hidden layer) could only learn a straight line.

4. **Scalar autograd is slow** but pedagogically clear: every gradient flows through individual Value objects. The next notebooks will use NumPy and then PyTorch tensors for efficiency.

5. **Parameter count**: MLP(2, [8, 8, 1]) has (2×8+8) + (8×8+8) + (8×1+1) = 24 + 72 + 9 = 105 parameters — enough for a toy problem, too few for real data.